# CelebAMask-HQ CPU BG / HAIR / FACE 5k Preprocess And Package

Use this notebook with a CPU runtime. It builds the 5k `BG / HAIR / FACE`
processed dataset, verifies the metadata, and creates both the train/val shard
package and the held-out test eval package on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CelebMaskHQ_Colab')
REPO_DRIVE = DRIVE_ROOT / 'repo'
RAW_DRIVE = DRIVE_ROOT / 'raw'
PROCESSED_DRIVE_ROOT = DRIVE_ROOT / 'processed'
PROJECT_LOCAL = Path('/content/project')

PROCESSED_NAME = 'processed_celebmaskhq_bghairface_5k'
TRAINVAL_PACKAGE_NAME = 'processed_celebmaskhq_bghairface_5k_trainval'
TRAINVAL_OUTPUT_NAME = 'processed_celebmaskhq_bghairface_5k_trainval_shards'
EVAL_PACKAGE_NAME = 'processed_celebmaskhq_bghairface_5k_test_eval'
EVAL_OUTPUT_NAME = 'processed_celebmaskhq_bghairface_5k_test_eval_package'

PROCESSED_DRIVE = PROCESSED_DRIVE_ROOT / PROCESSED_NAME
TRAINVAL_OUTPUT_DRIVE = PROCESSED_DRIVE_ROOT / TRAINVAL_OUTPUT_NAME
EVAL_OUTPUT_DRIVE = PROCESSED_DRIVE_ROOT / EVAL_OUTPUT_NAME

LAYERED_SCHEME = 'bg_hair_face'
SAMPLE_LIMIT = 5000
PREVIEW_COUNT = 8
MAX_LAYERS = 3
SHARD_SIZE = 2000
MAX_EVAL_SAMPLES = 8
PROGRESS_EVERY = 100

FORCE_REBUILD_PROCESSED = False
RESUME_EXISTING_PROCESSED = True
FORCE_REBUILD_TRAINVAL_PACKAGE = False
RESUME_EXISTING_TRAINVAL_PACKAGE = True
FORCE_REBUILD_EVAL_PACKAGE = False

print('Repo on Drive:', REPO_DRIVE)
print('Raw data on Drive:', RAW_DRIVE)
print('Processed output:', PROCESSED_DRIVE)
print('Train/val package:', TRAINVAL_OUTPUT_DRIVE)
print('Test eval package:', EVAL_OUTPUT_DRIVE)


In [ ]:
!pip install -U pillow numpy torch matplotlib


In [ ]:
import shutil

assert REPO_DRIVE.exists(), f'Missing repo folder on Drive: {REPO_DRIVE}'
assert RAW_DRIVE.exists(), f'Missing raw folder on Drive: {RAW_DRIVE}'
assert (RAW_DRIVE / 'CelebA-HQ-img').exists(), 'Missing raw/CelebA-HQ-img on Drive'
assert (RAW_DRIVE / 'CelebAMask-HQ-mask-anno').exists(), 'Missing raw/CelebAMask-HQ-mask-anno on Drive'

if PROJECT_LOCAL.exists():
    shutil.rmtree(PROJECT_LOCAL)
shutil.copytree(REPO_DRIVE, PROJECT_LOCAL)
PROCESSED_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Copied repo to', PROJECT_LOCAL)


In [ ]:
import subprocess

def run_checked(command, cwd, capture_output=True):
    printable = ' '.join(str(part) for part in command)
    print('Running command:', printable)
    if capture_output:
        result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
        print('RETURN CODE:', result.returncode)
        print('--- STDOUT ---')
        print(result.stdout)
        print('--- STDERR ---')
        print(result.stderr)
    else:
        result = subprocess.run(command, cwd=cwd, text=True)
        print('RETURN CODE:', result.returncode)
    if result.returncode != 0:
        raise RuntimeError('Command failed; see stdout/stderr above.')
    return result

pipeline_text = (PROJECT_LOCAL / 'preprocess_celebmaskhq' / 'pipeline.py').read_text(encoding='utf-8')
freshness_markers = ['LAYERED_SCHEMES', 'bg_hair_face', 'processed_splits.json', 'source_discovery_cache.json', 'layer_source_slots']
missing_markers = [marker for marker in freshness_markers if marker not in pipeline_text]
assert not missing_markers, f'Copied repo is stale; missing markers: {missing_markers}'
print('Pipeline freshness check passed.')


In [ ]:
import json
import shutil
import sys

if FORCE_REBUILD_PROCESSED and PROCESSED_DRIVE.exists():
    shutil.rmtree(PROCESSED_DRIVE)

processed_metadata_path = PROCESSED_DRIVE / 'metadata' / 'layered_samples.jsonl'

def processed_metadata_matches_target():
    if not processed_metadata_path.exists():
        return False
    try:
        mapping = json.loads((PROCESSED_DRIVE / 'metadata' / 'mapping.json').read_text(encoding='utf-8'))
        stats = json.loads((PROCESSED_DRIVE / 'metadata' / 'stats.json').read_text(encoding='utf-8'))
        rows = [json.loads(line) for line in processed_metadata_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    except Exception as exc:
        print('Existing metadata could not be read, will rebuild/resume:', repr(exc))
        return False
    mismatched_rows = [
        row for row in rows
        if row.get('layer_names') != ['BG', 'HAIR', 'FACE'] or int(row.get('layer_count', -1)) != 3
    ]
    if mismatched_rows:
        print('Existing metadata has rows outside the fixed BG/HAIR/FACE 3-layer contract.')
        print('First mismatched row:', json.dumps(mismatched_rows[0], indent=2, sort_keys=True)[:2000])
        return False
    return (
        mapping.get('layered_export', {}).get('scheme') == LAYERED_SCHEME
        and mapping.get('layered_export', {}).get('target_layer_names') == ['BG', 'HAIR', 'FACE']
        and int(stats.get('processed_sample_count', -1)) == SAMPLE_LIMIT
        and len(rows) == SAMPLE_LIMIT
    )

if FORCE_REBUILD_PROCESSED or not processed_metadata_matches_target():
    command = [
        sys.executable, '-u', 'scripts/build_processed_celebmaskhq.py',
        '--dataset-root', str(RAW_DRIVE),
        '--output-root', str(PROCESSED_DRIVE),
        '--layered-scheme', LAYERED_SCHEME,
        '--limit', str(SAMPLE_LIMIT),
        '--preview-count', str(PREVIEW_COUNT),
        '--progress-every', str(PROGRESS_EVERY),
    ]
    if RESUME_EXISTING_PROCESSED:
        command.append('--resume-existing')
    run_checked(command, PROJECT_LOCAL, capture_output=False)
else:
    print('Processed BG/HAIR/FACE 5k dataset already exists and matches the fixed 3-layer contract; skipping rebuild.')

assert processed_metadata_path.exists(), f'Missing layered metadata: {processed_metadata_path}'
print('Processed dataset ready at', PROCESSED_DRIVE)


In [ ]:
import json

mapping = json.loads((PROCESSED_DRIVE / 'metadata' / 'mapping.json').read_text(encoding='utf-8'))
stats = json.loads((PROCESSED_DRIVE / 'metadata' / 'stats.json').read_text(encoding='utf-8'))
rows = [json.loads(line) for line in (PROCESSED_DRIVE / 'metadata' / 'layered_samples.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]

assert mapping['layered_export']['scheme'] == LAYERED_SCHEME
assert mapping['layered_export']['target_layer_names'] == ['BG', 'HAIR', 'FACE']
assert stats['processed_sample_count'] == SAMPLE_LIMIT, stats['processed_sample_count']
mismatched_rows = [
    row for row in rows
    if row.get('layer_names') != ['BG', 'HAIR', 'FACE'] or int(row.get('layer_count', -1)) != 3
]
assert not mismatched_rows, mismatched_rows[0]

print('Metadata verification passed.')
print('Processed split counts:', stats['split_counts_processed_subset'])
print('Layer count histogram:', stats['layered_layer_count_histogram'])


In [ ]:
import sys

run_checked([
    sys.executable,
    'scripts/inspect_generic_layered_loader.py',
    '--processed-root', str(PROCESSED_DRIVE),
    '--split', 'train',
    '--batch-size', '1',
    '--num-batches', '1',
    '--max-layers', str(MAX_LAYERS),
    '--drop-warning-samples',
], PROJECT_LOCAL)


In [ ]:
import shutil
import sys

if FORCE_REBUILD_TRAINVAL_PACKAGE and TRAINVAL_OUTPUT_DRIVE.exists():
    shutil.rmtree(TRAINVAL_OUTPUT_DRIVE)

trainval_manifest_path = TRAINVAL_OUTPUT_DRIVE / 'package_manifest.json'
if not trainval_manifest_path.exists():
    command = [
        sys.executable, '-u', 'scripts/package_qwen_layered_trainval_shards.py',
        '--processed-root', str(PROCESSED_DRIVE),
        '--output-root', str(TRAINVAL_OUTPUT_DRIVE),
        '--package-name', TRAINVAL_PACKAGE_NAME,
        '--splits', 'train', 'val',
        '--shard-size', str(SHARD_SIZE),
        '--progress-every', str(PROGRESS_EVERY),
    ]
    if FORCE_REBUILD_TRAINVAL_PACKAGE:
        command.append('--force')
    elif RESUME_EXISTING_TRAINVAL_PACKAGE:
        command.append('--resume-existing')
    run_checked(command, PROJECT_LOCAL, capture_output=False)
else:
    print('Train/val shard package already exists; skipping rebuild.')

assert trainval_manifest_path.exists(), f'Missing train/val package manifest: {trainval_manifest_path}'
print('Train/val shard package ready at', TRAINVAL_OUTPUT_DRIVE)


In [ ]:
import json

trainval_manifest = json.loads((TRAINVAL_OUTPUT_DRIVE / 'package_manifest.json').read_text(encoding='utf-8'))
assert trainval_manifest['included_splits'] == ['train', 'val']
assert trainval_manifest['test_split_packaged'] == 0
assert trainval_manifest['data_shards'], 'No train/val data shards were produced.'
print('Train/val package verification passed.')
print('Split counts:', trainval_manifest['split_counts'])
print('Sample count:', trainval_manifest['sample_count'])
print('Data shard count:', len(trainval_manifest['data_shards']))


In [ ]:
import shutil
import sys

if FORCE_REBUILD_EVAL_PACKAGE and EVAL_OUTPUT_DRIVE.exists():
    shutil.rmtree(EVAL_OUTPUT_DRIVE)

eval_manifest_path = EVAL_OUTPUT_DRIVE / 'package_manifest.json'
if not eval_manifest_path.exists():
    command = [
        sys.executable, '-u', 'scripts/package_qwen_layered_eval_subset.py',
        '--processed-root', str(PROCESSED_DRIVE),
        '--output-root', str(EVAL_OUTPUT_DRIVE),
        '--package-name', EVAL_PACKAGE_NAME,
        '--split', 'test',
        '--max-samples', str(MAX_EVAL_SAMPLES),
        '--sample-strategy', 'spaced',
        '--progress-every', '1',
    ]
    if FORCE_REBUILD_EVAL_PACKAGE:
        command.append('--force')
    run_checked(command, PROJECT_LOCAL, capture_output=False)
else:
    print('Test eval package already exists; skipping rebuild.')

assert eval_manifest_path.exists(), f'Missing test eval package manifest: {eval_manifest_path}'
print('Test eval package ready at', EVAL_OUTPUT_DRIVE)


In [ ]:
import json

eval_manifest = json.loads((EVAL_OUTPUT_DRIVE / 'package_manifest.json').read_text(encoding='utf-8'))
assert eval_manifest['target_split'] == 'test'
assert eval_manifest['sample_count'] <= MAX_EVAL_SAMPLES
print('Test eval package verification passed.')
print('Selected test sample ids:', eval_manifest['selected_sample_ids'])


## Result

The 5k `BG / HAIR / FACE` processed dataset and both Colab packages are ready on Drive:

- `processed/processed_celebmaskhq_bghairface_5k`
- `processed/processed_celebmaskhq_bghairface_5k_trainval_shards`
- `processed/processed_celebmaskhq_bghairface_5k_test_eval_package`
